In [18]:
import rebound
import reboundx
import astropy.constants as constants
import astropy.units as units
import matplotlib.pyplot as plt
import numpy as np
import scipy as sc
from multiprocess import Pool
import celmech

import warnings
from joblib import Parallel, delayed
from celmech.miscellaneous import frequency_modified_fourier_transform as fmftc

In [26]:
def Nintegrate(run, yrs):
    # num = int((yrs-20)*1e6/(3.5e9/1000)) + 2
    num = int((yrs-20)*1e6/(3.5e9/1000)) 
    # print("Snapshot start is," ,num)
    tmax = -yrs*1e6
    tmin = -(yrs-20)*1e6
    Nsnaps = 1000
    interval = int(abs((tmax-tmin)/Nsnaps))
    
    sa = rebound.Simulationarchive("runs/run_" + str(run)+ ".bin")
    orig = sa[0]
    M0 = orig.particles['sun'].m
    earth_m = orig.particles['earth'].m/1.0123000370338813
    
    print("starting ", 'R_'+ str(run)+ '_'+ str(yrs) +'.bin ')
    sim = sa[num]
    sim.integrator_synchronize()

    # print("snapshot time is,", sim.t)
    
    # print("sim min time is,", tmin/1e6, "myr")
    # print("sim max time is,", tmax/1e6, "myr")
    print("sim centered around,", (tmax - ((tmax-tmin)/2))/1e6,"myr")

    sim.integrator = "WHCKL" 
    sim.ri_whfast.safe_mode = False
    sim.ri_whfast.corrector = 17
    sim.ri_whfast.keep_unsynchronized=True
    sim.dt = 4.062/365.25
 
    sim.save_to_file('samples/'+'R_'+ str(run)+ '_'+ str(yrs) +'.bin', interval=interval, delete_file=True)
    
    ps = sim.particles
    rebx = reboundx.Extras(sim)
    gr = rebx.load_force('gr_potential')
    rebx.add_force(gr)
    gr.params['c'] = 63240 # speed of light in AU/yr
    
    cf = rebx.load_force("quadrupole")
    rebx.add_force(cf)

    f = 0.8525
    mu_eff = f*(1*0.0123000370338813*(earth_m)**2)/(1.0123000370338813*earth_m)
    R = 0.0025696
    R_e = (constants.R_earth.to(units.au)).value
    ratio = R / R_e
    r = (ratio-((5.14/ 1.e9)*(abs(sim.t))))*R_e
    ps['earth'].params["Rcentral"] = r
    ps['earth'].params["mu_effcentral"] = mu_eff
    
    gh = rebx.load_force("gravitational_harmonics")
    rebx.add_force(gh)
    J2 = 2.25*1e-7
    sim.particles['sun'].params["J2"] = J2
    
    inc_sun = np.radians(7.155) # rad
    Omega_sun = np.radians(75.594) # rad
    R_eq_sun = (constants.R_sun.to(units.au)).value
    
    spin_axis_vector = [np.sin(inc_sun) * np.sin(Omega_sun), -np.sin(inc_sun) * np.cos(Omega_sun), np.cos(inc_sun)]
    sim.particles['sun'].params["Omega"] = spin_axis_vector
    sim.particles['sun'].params["R_eq"] = R_eq_sun
    
    times = np.linspace(sim.t, tmax, Nsnaps)
    rate = (-7.e14)
    
    Nout = len(sa)

    for i, time in enumerate(times):
        print(sim.t)
        sim.integrate(time)
        sim.particles[0].m = M0*np.exp(time / rate)
        r = (ratio-((5.14/ 1.e9)*(abs(time))))*R_e
        sim.particles['earth'].params["Rcentral"] = r
        sim.move_to_com

In [27]:
Nintegrate(1, 20)

starting  R_1_20.bin 
sim centered around, -10.0 myr
0.0
0.0
-20020.02002002002
-40040.04004004004
-60060.06006006006
-80080.08008008008
-100100.1001001001
-120120.12012012012
-140140.14014014014
-160160.16016016016
-180180.18018018018
-200200.2002002002
-220220.22022022022
-240240.24024024024
-260260.26026026026
-280280.2802802803
-300300.30030030024
-320320.3203203203
-340340.3403403403
-360360.36036036036
-380380.3803803803
-400400.4004004004
-420420.42042042036
-440440.44044044043
-460460.4604604604
-480480.4804804805
-500500.50050050044
-520520.5205205205
-540540.5405405406
-560560.5605605606
-580580.5805805805
-600600.6006006005
-620620.6206206207
-640640.6406406406
-660660.6606606606
-680680.6806806806
-700700.7007007007
-720720.7207207207
-740740.7407407407
-760760.7607607606
-780780.7807807808
-800800.8008008008
-820820.8208208208
-840840.8408408407
-860860.8608608609
-880880.8808808809
-900900.9009009008
-920920.9209209208
-940940.940940941
-960960.960960961
-980980.980980980

In [44]:
# def simulation(par):

#     run = par # unpack parameters
#     years = [1190, 3260]

#     for year in years:
#         Nintegrate(run, year)

In [ ]:
# %%time 

# with Pool() as pool:
#     Ngrid = 64
#     # Ngrid = 2
#     par_num = np.arange(1,1+Ngrid,1)
#     parameters = []
#     for run in par_num:
#         parameters.append(run)
#     results = pool.map(simulation,parameters)

/home/miniconda3/envs/ahekster/lib/python3.12/site-packages/rebound/simulationarchive.py:150: RuntimeWarning: You have to reset function pointers after creating a reb_simulation struct with a binary file.
  warnings.warn(message, RuntimeWarning)
/home/miniconda3/envs/ahekster/lib/python3.12/site-packages/rebound/simulationarchive.py:150: RuntimeWarning: You have to reset function pointers after creating a reb_simulation struct with a binary file.
  warnings.warn(message, RuntimeWarning)
/home/miniconda3/envs/ahekster/lib/python3.12/site-packages/rebound/simulationarchive.py:150: RuntimeWarning: You have to reset function pointers after creating a reb_simulation struct with a binary file.
  warnings.warn(message, RuntimeWarning)
/home/miniconda3/envs/ahekster/lib/python3.12/site-packages/rebound/simulationarchive.py:150: RuntimeWarning: You have to reset function pointers after creating a reb_simulation struct with a binary file.
  warnings.warn(message, RuntimeWarning)
/home/miniconda3

starting starting starting 

/home/miniconda3/envs/ahekster/lib/python3.12/site-packages/rebound/simulationarchive.py:150: RuntimeWarning: You have to reset function pointers after creating a reb_simulation struct with a binary file.
  warnings.warn(message, RuntimeWarning)
/home/miniconda3/envs/ahekster/lib/python3.12/site-packages/rebound/simulationarchive.py:150: RuntimeWarning: You have to reset function pointers after creating a reb_simulation struct with a binary file.
  warnings.warn(message, RuntimeWarning)


starting starting   

/home/miniconda3/envs/ahekster/lib/python3.12/site-packages/rebound/simulationarchive.py:150: RuntimeWarning: You have to reset function pointers after creating a reb_simulation struct with a binary file.
  warnings.warn(message, RuntimeWarning)


...


myr-1180.0sim centered around,myrsim centered around, 

/home/miniconda3/envs/ahekster/lib/python3.12/site-packages/rebound/simulationarchive.py:152: RuntimeWarning: The simulation might not be synchronized. You can manually synchronize it by calling sim.integrator_synchronize().
  warnings.warn("The simulation might not be synchronized. You can manually synchronize it by calling sim.integrator_synchronize().", RuntimeWarning)


 
-1180.0


/home/miniconda3/envs/ahekster/lib/python3.12/site-packages/rebound/simulationarchive.py:152: RuntimeWarning: The simulation might not be synchronized. You can manually synchronize it by calling sim.integrator_synchronize().
  warnings.warn("The simulation might not be synchronized. You can manually synchronize it by calling sim.integrator_synchronize().", RuntimeWarning)
/home/miniconda3/envs/ahekster/lib/python3.12/site-packages/rebound/simulationarchive.py:152: RuntimeWarning: The simulation might not be synchronized. You can manually synchronize it by calling sim.integrator_synchronize().
  warnings.warn("The simulation might not be synchronized. You can manually synchronize it by calling sim.integrator_synchronize().", RuntimeWarning)
/home/miniconda3/envs/ahekster/lib/python3.12/site-packages/rebound/simulationarchive.py:152: RuntimeWarning: The simulation might not be synchronized. You can manually synchronize it by calling sim.integrator_synchronize().
  warnings.warn("The simu

sim centered around,  sim centered around,
 sim centered around,myr sim centered around, 

sim centered around,sim centered around,sim centered around, sim centered around,sim centered around,myrmyr1-1180.0   -1180.0 -1180.0 sim centered around, 0sim centered around,sim centered around,-1180.0-1180.0  sim centered around,-1180.0myr
sim centered around, 
  myrmyrsim centered around, 1sim centered around, sim centered around,myrmyrsim centered around, -1180.0 -1180.01
 myr   myr  

0
 -1180.00-1180.0 myr-1180.0 myr 
-1180.0   -1180.0 -1180.0  
myr -1180.0-1180.0
 
myr
 -1180.00 

0-1180.0  -1180.0
 -1180.0
-1180.0-1180.0-1180.0
-1180.0-1180.00
myr 
 myr0 
myr-1180.00
-1180.0 myr1-1180.0  myr-1180.0myr
-1180.0  -1180.0
-1180.0 
-1180.0
000 myr-1180.0 myr 00     

0myr1myr0

1myr

 myr 

 myrmyr
 
0 myrmyr 0 myr 0
1


myr myr
01myr

myrmyrmyrmyrmyr
1




1
myr0
myr10
myr
myr0

myr0
myr
myr0

myr0

11
myr

1
01


1

0
1
1


0
0



00

10
0
0
11
0


0
1


0
0

0
0000

101

001

1
0
01

0
01
